In [1]:
import os


In [2]:
%pwd

'/mnt/d/resume_projects/flight_fare_prediction/research'

In [3]:
os.chdir("../")
%pwd

'/mnt/d/resume_projects/flight_fare_prediction'

In [4]:
# update entity
from dataclasses import dataclass
from pathlib import Path

@dataclass
class FeatureEngineeringConfig:
    root_dir: Path
    engineered_train_file_name: Path
    engineered_test_file_name: Path
    target_column: str
    replace_destinations: dict

In [5]:
@dataclass
class FeatureEngineeringArtifact:
    engineered_train_file_name: Path
    engineered_test_file_name: Path

In [6]:
from src.flight_price_prediction.constants import *
from src.flight_price_prediction.utils.common import *
from src.flight_price_prediction.entity.config_entity import FeatureEngineeringConfig
from pathlib import Path

In [7]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_feature_engineering_config(self) -> FeatureEngineeringConfig:
        config = self.config.feature_enginnering
        params = self.params.feature_enginnering

        engineered_train_file_name = Path(self.config.engineered_train_file_name)
        engineered_test_file_name = Path(self.config.engineered_test_file_name)
        target_column = str(params.target_column)
        stop_map = dict(params.stop_map)
        replace_destinations = dict(params.replace_destinations)

        create_directories([Path(config.root_dir)],engineered_train_file_name.parent,engineered_test_file_name.parent)

        return FeatureEngineeringConfig(
            root_dir= Path(config.root_dir),
            engineered_train_file_name = engineered_train_file_name,
            engineered_test_file_name = engineered_test_file_name,
            target_column = target_column,
            stop_map = stop_map,
            replace_destinations = replace_destinations

        )

In [8]:
from src.flight_price_prediction.utils.feature_engineering_utils import *
from src.flight_price_prediction.entity.config_entity import FeatureEngineeringConfig
from src.flight_price_prediction.entity.artifact_entity import FeatureEngineeringArtifact,DataValidationArtifact
from src.flight_price_prediction.logging.logger import logging
from src.flight_price_prediction.exception.exception import CustomException
import pandas as pd
import numpy as np


In [9]:
# feature engineering component
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, config: FeatureEngineeringConfig):
        self.config = config
        self.transformer = ColumnTransformer(
            transformer=[
               ('airline', 'passthrough', ['Airline']),
                ('source', 'passthrough', ['Source']),
                ('destination', DestinationNormalizer(self.config.replace_destinations), ['Destination']),
                ('total_stops', TotalStopsTransformer(self.config.stop_map), ['Total_Stops']),
                ('route_interaction', RouteInteractionTransformer(self.config.replace_destinations), ['Source', 'Destination']),
                ('date', DateFeatureExtractor(), ['Date_of_Journey']),
                ('duration', DurationMinutesTransformer(), ['Duration']),
                ('dep_time', TimeCyclicTransformer(), ['Dep_Time']),
                ('arrival_time', TimeCyclicTransformer(), ['Arrival_Time']), 
                ('duration_times_stops', DurationStopsInteractionTransformer(), ['Duration', 'Total_Stops']),
            ],
             sparse_threshold=0,
             remainder='drop',
        )
    def fit(self, X, y=None):
        self.transformer.fit(X)
        return self
    def transform(self, X):
        transformed = self.transformer.transform(X)
        columns =['airline',
            'source',
            'destination',
            'total_stops',
            'route_interaction',
            'month',
            'day',
            'duration_to_minutes',
            'dep_sin',
            'dep_cos',
            'arrival_sin',
            'arrival_cos',
            'duration_times_stops'
        ]
        df = pd.DataFrame(transformed, columns=columns)
        return df




In [10]:
class FeatureEngineering:
    def __init__(self, 
                 feature_engineering_config: FeatureEngineeringConfig,
                 data_validation_artifact: DataValidationArtifact):
        try:
            self.config = feature_engineering_config
            self.data_validation_artifact = data_validation_artifact
            self.feature_transformer = FeatureEngineeringTransformer(self.config)
        except Exception as e:
            raise CustomException(e, sys)
        
    @staticmethod
    def read_data(file_path) -> pd.DataFrame:
        try:
            return pd.read_csv(file_path)
        except Exception as e:
            raise CustomException(e, sys)
        
    def _clean_raw_dataframe(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        dataframe = dataframe.rename(columns={
            'Airline': 'Airline',
            'Date_of_Journey': 'Date_of_Journey',
            'Source': 'Source',
            'Destination': 'Destination',
            'Route': 'Route',
            'Dep_Time': 'Dep_Time',
            'Arrival_Time': 'Arrival_Time',
            'Duration': 'Duration',
            'Total_Stops': 'Total_Stops',
            'Additional_Info': 'Additional_Info',
            'Price': 'Price'
        })
        dataframe = dataframe[dataframe['Duration'] != '5m']
        dataframe = dataframe.drop_duplicates()
        return dataframe
    

    def initiate_feature_engineering(self) -> FeatureEngineeringArtifact:
        try:
            logging.info('Starting feature engineering on validated datasets')
            train_df = FeatureEngineering.read_data(self.data_validation_artifact.valid_train_file_name)
            test_df = FeatureEngineering.read_data(self.data_validation_artifact.valid_test_file_name)
            
            logging.info('cleaning data and renaming columns started')
            train_df = self._clean_raw_dataframe(train_df)
            test_df = self._clean_raw_dataframe(test_df)
            
            logging.info('dividing target column as train_target ans test_target before performing featuire engineering')
            train_target = pd.to_numeric(train_df['Price'], errors='coerce')
            test_target = pd.to_numeric(test_df['Price'], errors='coerce')

            logging.info("apllying feature_transformer for train and test sets")
            train_engineered = self.feature_transformer.fit_transform(train_df)
            test_engineered = self.feature_transformer.transform(test_df)
            
            logging.info('adding target column again to train and test engineered sets')
            train_engineered['price'] = train_target.values
            test_engineered['price'] = test_target.values
            
            logging.info('Dropping null values')
            train_engineered = train_engineered.dropna(subset=['price'])
            test_engineered = test_engineered.dropna(subset=['price'])

            logging.info("saving engineered train and test data into artifacts")
            save_data(train_engineered, Path(self.config.engineered_train_file_name))
            save_data(test_engineered, Path(self.config.engineered_test_file_name))
            
            logging.info(f'Feature engineered train file saved to {self.config.engineered_train_file_name}')
            logging.info(f'Feature engineered test file saved to {self.config.engineered_test_file_name}')

            return FeatureEngineeringArtifact(
                engineered_train_file_name=Path(self.config.engineered_train_file_name),
                engineered_test_file_name=Path(self.config.engineered_test_file_name),
            )
        except Exception as e:
            raise CustomException(e, sys)


             






In [11]:
#pipleine
import sys
from src.flight_price_prediction.config.configuration import ConfigurationManager
from src.flight_price_prediction.components.feature_engineering import FeatureEngineering
from src.flight_price_prediction.entity.artifact_entity import DataValidationArtifact, FeatureEngineeringArtifact
from src.flight_price_prediction.exception.exception import CustomException
from src.flight_price_prediction.logging.logger import logging

STAGE_NAME = "Feature Engineering Stage"

class FeatureEngineeringTrainingPipeline:
    def __init__(self, config: ConfigurationManager, data_validation_artifact: DataValidationArtifact):
        try:
            self.config = config
            self.data_validation_artifact = data_validation_artifact
        except Exception as e:
            raise CustomException(e, sys)

    def initiate_feature_engineering(self) -> FeatureEngineeringArtifact:
        try:
            logging.info(f'>>>> stage {STAGE_NAME} started <<<<')
            feature_engineering_config = self.config.get_feature_engineering_config()
            feature_engineering = FeatureEngineering(
                feature_engineering_config=feature_engineering_config,
                data_validation_artifact=self.data_validation_artifact,
            )
            artifact = feature_engineering.initiate_feature_engineering()
            logging.info(f'>>>> stage {STAGE_NAME} completed <<<<')
            return artifact
        except Exception as e:
            raise CustomException(e, sys)

if __name__ == '__main__':
    try:
        logging.info(f'>>>> stage {STAGE_NAME} started <<<<')
        config = ConfigurationManager()
        from src.flight_price_prediction.pipeline.data_validation_pipeline import DataValidationTrainingPipeline
        from src.flight_price_prediction.pipeline.data_ingestion_pipeline import DataIngestionTrainingPipeline
        ingestion_pipeline = DataIngestionTrainingPipeline(config=config)
        ingestion_artifact = ingestion_pipeline.initiate_data_ingestion()
        validation_pipeline = DataValidationTrainingPipeline(config=config, data_ingestion_artifact=ingestion_artifact)
        validation_artifact = validation_pipeline.initiate_data_validation()
        pipeline = FeatureEngineeringTrainingPipeline(config=config, data_validation_artifact=validation_artifact)
        pipeline.initiate_feature_engineering()
        logging.info(f'>>>> stage {STAGE_NAME} completed <<<<')
    except Exception as e:
        raise CustomException(e, sys)

[2026-06-09 16:44:01,251: INFO: 4213946113: >>>> stage Feature Engineering Stage started <<<<]
[2026-06-09 16:44:01,262: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/config/config.yaml loaded succesfully ]
[2026-06-09 16:44:01,277: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/params/params.yaml loaded succesfully ]
[2026-06-09 16:44:01,288: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/schema/schema.yaml loaded succesfully ]
[2026-06-09 16:44:01,295: INFO: common: created directory at: artifacts]
[2026-06-09 16:44:02,609: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-09 16:44:03,099: INFO: data_ingestion: Connecting to MongoDB at: mongodb+srv://p...]
[2026-06-09 16:44:04,005: INFO: data_ingestion: Successfully connected to MongoDB.]
[2026-06-09 16:44:12,979: INFO: common: Data saved to: artifacts/data_ingestion/feature_store/flight_fare.csv]
[2026-06-09 16:44:12,980: INFO: data_ingesti